<a href="https://colab.research.google.com/github/rathod625/Registration-Form/blob/main/AI_Resume_Analyser.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install streamlit pdfplumber scikit-learn nltk plotly -q

# Download NLTK
import nltk
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('punkt_tab', quiet=True)

# Write app
app = """
import streamlit as st
import pdfplumber, re, json, nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import plotly.graph_objects as go
import plotly.express as px

nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('punkt_tab', quiet=True)

st.set_page_config(page_title="AI Resume Analyser", layout="wide")

def extract_pdf(f):
    t = ""
    with pdfplumber.open(f) as pdf:
        for p in pdf.pages:
            x = p.extract_text()
            if x: t += x + "\\n"
    return t.strip()

def clean(text):
    lem = WordNetLemmatizer()
    sw = set(stopwords.words('english'))
    text = re.sub(r'[^a-z\\s]', ' ', text.lower())
    return " ".join([lem.lemmatize(w) for w in word_tokenize(text) if w not in sw and len(w)>2])

def ats_score(r, j):
    v = TfidfVectorizer()
    m = v.fit_transform([r, j])
    return round(float(cosine_similarity(m[0:1], m[1:2])[0][0]) * 100, 1)

def get_kw(text, n=35):
    v = TfidfVectorizer(max_features=n, stop_words='english')
    try:
        v.fit_transform([text])
        return list(v.get_feature_names_out())
    except: return []

def gaps(r, j):
    rk, jk = set(get_kw(r)), set(get_kw(j))
    return sorted(rk & jk), sorted(jk - rk)

def tips(score, missing):
    t = []
    if score < 40: t.append("Very low score. Rewrite resume targeting this JD.")
    elif score < 60: t.append("Moderate match. Add more JD keywords.")
    elif score < 80: t.append("Good match! Small additions will perfect it.")
    else: t.append("Excellent score! Resume is well optimised.")
    if missing: t.append("Add these missing keywords: " + ", ".join(missing[:6]))
    t.append("Use exact job title from JD in your resume headline.")
    t.append("Quantify achievements - e.g. Improved performance by 30 percent.")
    t.append("Avoid tables and graphics - ATS cannot read them.")
    return t

st.title("AI Resume Analyser")
st.markdown("Powered by NLP - TF-IDF - Cosine Similarity")
st.markdown("By Raju | Internship Project 2026")
st.markdown("---")

c1, c2 = st.columns(2)
with c1:
    st.subheader("Upload Resume PDF")
    pdf = st.file_uploader("Choose PDF file", type=["pdf"])
with c2:
    st.subheader("Paste Job Description")
    jd = st.text_area("Paste JD here", height=200)

st.markdown("---")

if st.button("ANALYSE MY RESUME", use_container_width=True, type="primary"):
    if not pdf: st.error("Upload resume PDF"); st.stop()
    if not jd.strip(): st.error("Paste job description"); st.stop()

    with st.spinner("Analysing..."):
        raw = extract_pdf(pdf)
        if not raw: st.error("Cannot read PDF. Use text-based PDF."); st.stop()
        score = ats_score(clean(raw), clean(jd))
        matched, missing = gaps(raw, jd)
        suggestions = tips(score, missing)

    st.success("Analysis Complete!")
    st.markdown("---")

    color = "#00ff88" if score >= 75 else "#ffaa00" if score >= 50 else "#ff4444"
    a, b, c, d = st.columns(4)
    a.metric("ATS SCORE", str(score) + "%")
    b.metric("MATCHED", str(len(matched)) + " keywords")
    c.metric("MISSING", str(len(missing)) + " keywords")
    d.metric("WORD COUNT", str(len(raw.split())))

    fig = go.Figure(go.Indicator(
        mode="gauge+number", value=score,
        title={"text": "ATS Score"},
        number={"suffix": "%"},
        gauge={
            "axis": {"range": [0,100]},
            "bar": {"color": color},
            "steps": [
                {"range": [0,40], "color": "#ffeeee"},
                {"range": [40,70], "color": "#ffffee"},
                {"range": [70,100], "color": "#eeffee"}
            ]
        }
    ))
    fig.update_layout(height=280)
    st.plotly_chart(fig, use_container_width=True)

    st.markdown("---")
    k1, k2 = st.columns(2)
    with k1:
        st.subheader("Matched Keywords")
        for k in matched: st.success(k)
    with k2:
        st.subheader("Missing Keywords")
        for k in missing[:15]: st.error(k)

    st.markdown("---")
    st.subheader("Improvement Suggestions")
    for tip in suggestions: st.info(tip)

    st.markdown("---")
    st.download_button(
        "Download Report JSON",
        json.dumps({"score": score, "matched": matched, "missing": missing, "tips": suggestions}, indent=2),
        "report.json", "application/json", use_container_width=True
    )
"""

with open("app.py", "w") as f:
    f.write(app)

# Run with colab port forwarding
import subprocess, time
subprocess.run(["pkill", "-f", "streamlit"], capture_output=True)
time.sleep(2)
subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"])
time.sleep(4)
print("DONE! Now go to Colab menu: Runtime → Ports → click port 8501")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 693.9 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 43.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 41.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 59.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 42.8 MB/s eta 0:00:00
DONE! Now go to Colab menu: Runtime → Ports → click port 8501


In [3]:
from google.colab.output import eval_js
print(eval_js("google.colab.kernel.proxyPort(8501)"))

https://8501-m-s-kkb-usw4a2-vq0efzakfthj-a.us-west4-2.prod.colab.dev


In [4]:
import subprocess, time

subprocess.run(["pkill", "-f", "streamlit"], capture_output=True)
time.sleep(2)

subprocess.Popen([
    "streamlit", "run", "app.py",
    "--server.port", "8501",
    "--server.headless", "true",
    "--server.enableCORS", "false",
    "--server.enableXsrfProtection", "false",
    "--server.enableWebsocketCompression", "false",
    "--server.allowRunOnSave", "false"
])

time.sleep(6)

from google.colab.output import eval_js
url = eval_js("google.colab.kernel.proxyPort(8501)")
print("Click this link: " + str(url))

Click this link: https://8501-m-s-kkb-usw4a2-vq0efzakfthj-a.us-west4-2.prod.colab.dev
